In [20]:
%pip install pandas duckdb


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Set up and load RAW tables

In [21]:
import duckdb
import pandas as pd
import os
from pathlib import Path

In [22]:
# Connect to duck db 

if os.path.exists('ecommerce.duckdb'):
    # We try to connect in-memory first to ensure we aren't locking anything
    duckdb.connect().close()

# 1. Connect (This will create a FRESH, valid database file)
con = duckdb.connect('ecommerce.duckdb')

# sanity check
for f in ["List of Orders.csv", "Order Details.csv", "Sales target.csv"]:
    p = Path("data") / f
    assert p.exists(), f"Missing file: {p}"

In [23]:
# Create schemas and load raw tables
con.execute("CREATE SCHEMA IF NOT EXISTS raw;")
con.execute("CREATE SCHEMA IF NOT EXISTS stg;") #staging layer
con.execute("CREATE SCHEMA IF NOT EXISTS clean;") 

# Load CSVs into raw tables
con.execute("CREATE OR REPLACE TABLE raw.orders AS SELECT * FROM read_csv_auto('data/List of Orders.csv');")
con.execute("CREATE OR REPLACE TABLE raw.order_details AS SELECT * FROM read_csv_auto('data/Order Details.csv');")
con.execute("CREATE OR REPLACE TABLE raw.sales_target AS SELECT * FROM read_csv_auto('data/Sales target.csv');")

con.sql("SHOW TABLES FROM raw").show()


┌───────────────┐
│     name      │
│    varchar    │
├───────────────┤
│ order_details │
│ orders        │
│ sales_target  │
└───────────────┘



In [24]:
# Helper to run SQL and return a df
def q(sql):
    return con.sql(sql).to_df()

In [25]:
# Row counts
q("""
SELECT 'raw.orders' AS table_name, COUNT(*) AS rows FROM raw.orders
UNION ALL
SELECT 'raw.order_details', COUNT(*) FROM raw.order_details
UNION ALL
SELECT 'raw.sales_target', COUNT(*) FROM raw.sales_target;
""")

,table_name,rows
0,raw.orders,560
1,raw.order_details,1500
2,raw.sales_target,36


## 2. STAGING views

In [26]:
q(""" DESCRIBE raw.orders """)

,column_name,column_type,null,key,default,extra
0,Order ID,VARCHAR,YES,None,None,None
1,Order Date,DATE,YES,None,None,None
2,CustomerName,VARCHAR,YES,None,None,None
3,State,VARCHAR,YES,None,None,None
4,City,VARCHAR,YES,None,None,None


In [27]:
# staging order

con.execute("""
CREATE OR REPLACE VIEW stg.orders AS
SELECT
    TRIM("Order ID") AS order_id,
    CAST("Order Date" AS DATE) AS order_date, 
    TRIM("CustomerName") AS customer_name,
    TRIM("State") AS state, 
    TRIM("City") AS city
FROM raw.orders
WHERE "Order ID" IS NOT NULL;
""")

q(""" DESCRIBE stg.orders""")

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,order_date,DATE,YES,None,None,None
2,customer_name,VARCHAR,YES,None,None,None
3,state,VARCHAR,YES,None,None,None
4,city,VARCHAR,YES,None,None,None


In [28]:
q(""" DESCRIBE raw.order_details """)

,column_name,column_type,null,key,default,extra
0,Order ID,VARCHAR,YES,None,None,None
1,Amount,DOUBLE,YES,None,None,None
2,Profit,DOUBLE,YES,None,None,None
3,Quantity,BIGINT,YES,None,None,None
4,Category,VARCHAR,YES,None,None,None
5,Sub-Category,VARCHAR,YES,None,None,None


In [29]:
# staging order details
con.execute("""
CREATE OR REPLACE VIEW stg.order_details AS
SELECT
    TRIM("Order ID") AS order_id, 
    "Amount" :: DOUBLE AS amount,
    "Profit" :: DOUBLE AS profit, 
    "Quantity" :: INTEGER AS quantity, 
    TRIM("Category") AS category, 
    TRIM("Sub-Category") AS sub_category
FROM raw.order_details;
""")

q(""" DESCRIBE stg.order_details""")

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,amount,DOUBLE,YES,None,None,None
2,profit,DOUBLE,YES,None,None,None
3,quantity,INTEGER,YES,None,None,None
4,category,VARCHAR,YES,None,None,None
5,sub_category,VARCHAR,YES,None,None,None


In [30]:
q(""" DESCRIBE raw.sales_target """)

,column_name,column_type,null,key,default,extra
0,Month of Order Date,VARCHAR,YES,None,None,None
1,Category,VARCHAR,YES,None,None,None
2,Target,DOUBLE,YES,None,None,None


In [31]:
con.execute("""
CREATE OR REPLACE VIEW stg.sales_target AS
SELECT
    STRPTIME("Month of Order Date", '%b-%Y'):: DATE AS month, 
    TRIM("Category") AS category, 
    "Target":: DOUBLE AS target
FROM raw.sales_target
""")

q(""" DESCRIBE stg.sales_target """)

,column_name,column_type,null,key,default,extra
0,month,DATE,YES,None,None,None
1,category,VARCHAR,YES,None,None,None
2,target,DOUBLE,YES,None,None,None


In [32]:
# Validate staging outputs
q("""
SELECT 'stg.orders' AS table_name, COUNT(*) AS rows FROM stg.orders
UNION ALL
SELECT 'stg.order_details', COUNT(*) FROM stg.order_details
UNION ALL
SELECT 'stg.sales_target', COUNT(*) FROM stg.sales_target;
""")


,table_name,rows
0,stg.orders,500
1,stg.order_details,1500
2,stg.sales_target,36


In [33]:
# Two key integrity checks 

# order_id uniqueness in orders
q("""
SELECT order_id, COUNT(*) AS n
FROM stg.orders
GROUP BY 1
HAVING COUNT(*) > 1;
""")


,order_id,n


In [34]:
# orphan detail lines (details without header)
q("""
SELECT COUNT(*) AS orphan_lines
FROM stg.order_details d
LEFT JOIN stg.orders o USING(order_id)
WHERE o.order_id IS NULL;
""")

,orphan_lines
0,0


## Create clean fact table

A fact table is thw table you aggrgate to compute the KPIs

In [35]:
con.execute(""" 
CREATE OR REPLACE TABLE clean.fact_order_lines AS 
SELECT
    o.order_id, 
    o.order_date,
    DATE_TRUNC('month', o.order_date):: DATE AS order_month,        
    o.customer_name, 
    o.state, 
    o.city, 
    
    d.amount AS revenue, 
    d.profit, 
    d.quantity,
    d.category, 
    d.sub_category
FROM stg.order_details d 
    JOIN stg.orders o  -- Check what type of join, order details has repeated order ID
    USING(order_id)
""")

# The DATE_TRUNC('month', o.order_date):: DATE AS order_month
# we do it to have the month of each order so its easier to group them 

        

### Validations

In [36]:
# Row count, should be 1500
q("SELECT COUNT(*) FROM clean.fact_order_lines")

,count_star()
0,1500


In [37]:
# Sanity checks 
q("""
SELECT
  MIN(order_date) AS min_date,
  MAX(order_date) AS max_date, 
  SUM(revenue) AS total_revenue,
  SUM(profit) AS total_profit
FROM clean.fact_order_lines;
""")


,min_date,max_date,total_revenue,total_profit
0,2018-04-01,2019-03-31,431502.0,23955.0


In [38]:
# Chek for missing values in key fields
q("""
SELECT
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
  SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) AS null_order_date,
  SUM(CASE WHEN revenue IS NULL THEN 1 ELSE 0 END) AS null_revenue
FROM clean.fact_order_lines;
""")

,null_order_id,null_order_date,null_revenue
0,0.0,0.0,0.0


## 3. KPIs

### 3.1 Monthly revenue vs target

In [43]:
con.execute("""
CREATE OR REPLACE TABLE clean.kpi_monthly_category AS 
            SELECT order_month,
            category,
            SUM(revenue) AS revenue,
            SUM(profit) AS profit,
            SUM(quantity) AS units, 
            COUNT(DISTINCT order_id) AS num_orders
    FROM clean.fact_order_lines
    GROUP BY order_month, category
""")

In [49]:
con.execute("""
CREATE OR REPLACE TABLE clean.kpi_monthly_category_vs_target AS 
        SELECT k.order_month,
            k.category,
            k.revenue,
            k.profit,
            k.units,
            k.num_orders, 
            t.target, 
            (k.revenue - t.target) AS variance,
            CASE 
                WHEN (t.target = 0 OR t.target IS NULL) THEN NULL
                ELSE (k.revenue - t.target)/t.target
            END AS variance_percentage
    FROM clean.kpi_monthly_category k 
        LEFT JOIN stg.sales_target t 
            ON k.order_month = t.month AND k.category = t.category 

""")

In [50]:
q("""
SELECT
  COUNT(*) AS rows,
  SUM(CASE WHEN target IS NULL THEN 1 ELSE 0 END) AS missing_targets
FROM clean.kpi_monthly_category_vs_target;
""")


,rows,missing_targets
0,36,36.0


In [52]:
q("""
SELECT table_schema, table_name, table_type
FROM information_schema.tables
WHERE table_schema IN ('raw','stg','clean')
ORDER BY table_schema, table_type, table_name;
""")


,table_schema,table_name,table_type
0,clean,fact_order_lines,BASE TABLE
1,clean,kpi_monthly_category,BASE TABLE
2,clean,kpi_monthly_category_vs_target,BASE TABLE
3,raw,order_details,BASE TABLE
4,raw,orders,BASE TABLE
5,raw,sales_target,BASE TABLE
6,stg,order_details,VIEW
7,stg,orders,VIEW
8,stg,sales_target,VIEW


In [57]:
q(""" SELECT * FROM clean.kpi_monthly_category_vs_target LIMIT 5""")

,order_month,category,revenue,profit,units,num_orders,target,variance,variance_percentage
0,2018-07-01,Clothing,2981.0,-48.0,142.0,21,NaN,NaN,NaN
1,2019-02-01,Furniture,16262.0,2168.0,118.0,21,NaN,NaN,NaN
2,2018-11-01,Electronics,16651.0,3938.0,105.0,18,NaN,NaN,NaN
3,2019-02-01,Clothing,9569.0,1822.0,312.0,42,NaN,NaN,NaN
4,2018-09-01,Electronics,7207.0,-910.0,56.0,12,NaN,NaN,NaN
